In [2]:
from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# Expected experiment structure
# ------------------------------------------------------------

QUICK_SWEEP_ROOTS = {
    "splitnet_attn": Path("recent_analysis/physics_tests"),
    "splitnet": Path("recent_analysis_split_na/physics_tests"),
    "attn_unet": Path("recent_analysis_unet/physics_tests"),
    "unet": Path("recent_analysis_unet_na/physics_tests"),
    "prof_unet": Path("recent_analysis_prof_unet/physics_tests"),
}

QUICK_DATASET_MODES = ["border", "border_pressure", "fixed"]

QUICK_DARCY_WEIGHTS = [
    0.0,
    0.001,
    0.01,
    0.1,
    1.0,
    10.0,
]


OFFICIAL_DARCY_ROOT = Path("official_darcy/final")

OFFICIAL_DARCY_MODEL_TYPES = [
    "splitnet_attn",
    "prof_unet",
    "attn_unet",
    "splitnet",
    "unet",
]

OFFICIAL_DARCY_DATASET_MODES = ["fixed", "border"]

OFFICIAL_DARCY_WEIGHTS = [
    5.0,
    0.1,
    1.0,
    10.0,
]


BASELINE_FULL_ROOT = Path("baseline_full/final")

BASELINE_FULL_MODEL_TYPES = [
    "splitnet_attn",
    "attn_unet",
    "prof_unet",
]

BASELINE_FULL_DATASET_MODES = [
    "fixed",
    "border",
]


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def safe_weight(w):
    return str(w).replace(".", "p")


def check_file(path):
    return Path(path).exists()


def add_expected(rows, category, model_type, dataset_mode, training_mode, darcy_weight, base_path, run_name):
    base_path = Path(base_path)

    expected = {
        "category": category,
        "model_type": model_type,
        "dataset_mode": dataset_mode,
        "training_mode": training_mode,
        "darcy_weight": darcy_weight,
        "run_name": run_name,

        "best_state_path": str(base_path / f"{run_name}_best_state.pt"),
        "final_state_path": str(base_path / f"{run_name}_final_state.pt"),
        "history_csv_path": str(base_path / f"{run_name}_history.csv"),
        "history_pt_path": str(base_path / f"{run_name}_history.pt"),
        "summary_json_path": str(base_path / f"{run_name}_summary.json"),
    }

    expected["has_best_state"] = check_file(expected["best_state_path"])
    expected["has_final_state"] = check_file(expected["final_state_path"])
    expected["has_history_csv"] = check_file(expected["history_csv_path"])
    expected["has_history_pt"] = check_file(expected["history_pt_path"])
    expected["has_summary_json"] = check_file(expected["summary_json_path"])

    expected["complete_core"] = (
        expected["has_best_state"]
        and expected["has_history_csv"]
    )

    rows.append(expected)


# ------------------------------------------------------------
# Build expected file table
# ------------------------------------------------------------

rows = []

# 1. Quick thin physics-limited Darcy sweeps
for model_type, root in QUICK_SWEEP_ROOTS.items():
    for dataset_mode in QUICK_DATASET_MODES:
        for w in QUICK_DARCY_WEIGHTS:
            sw = safe_weight(w)
            run_name = f"{dataset_mode}_physics_limited_{model_type}_darcy_{sw}"
            base_path = root / dataset_mode

            add_expected(
                rows=rows,
                category="quick_thin_darcy_sweep",
                model_type=model_type,
                dataset_mode=dataset_mode,
                training_mode="physics_limited",
                darcy_weight=w,
                base_path=base_path,
                run_name=run_name,
            )

# 2. Official dense/longer Darcy models
for model_type in OFFICIAL_DARCY_MODEL_TYPES:
    for dataset_mode in OFFICIAL_DARCY_DATASET_MODES:
        for w in OFFICIAL_DARCY_WEIGHTS:
            sw = safe_weight(w)
            run_name = f"{dataset_mode}_physics_limited_{model_type}_darcy_{sw}"

            add_expected(
                rows=rows,
                category="official_darcy",
                model_type=model_type,
                dataset_mode=dataset_mode,
                training_mode="physics_limited",
                darcy_weight=w,
                base_path=OFFICIAL_DARCY_ROOT,
                run_name=run_name,
            )

# 3. Baseline full no-Darcy models
for model_type in BASELINE_FULL_MODEL_TYPES:
    for dataset_mode in BASELINE_FULL_DATASET_MODES:
        run_name = f"{dataset_mode}_{model_type}_baseline_full_nodarcy"

        add_expected(
            rows=rows,
            category="baseline_full_nodarcy",
            model_type=model_type,
            dataset_mode=dataset_mode,
            training_mode="baseline_full",
            darcy_weight=0.0,
            base_path=BASELINE_FULL_ROOT,
            run_name=run_name,
        )

df = pd.DataFrame(rows)

# ------------------------------------------------------------
# Print useful summaries
# ------------------------------------------------------------

print("Total expected runs:", len(df))
print("Complete core runs:", df["complete_core"].sum())
print("Missing/incomplete runs:", (~df["complete_core"]).sum())

print("\nBy category:")
print(df.groupby("category")["complete_core"].agg(["sum", "count"]))

print("\nMissing core files:")
missing = df[~df["complete_core"]].copy()

if len(missing) == 0:
    print("None. All expected core files were found.")
else:
    display_cols = [
        "category",
        "model_type",
        "dataset_mode",
        "training_mode",
        "darcy_weight",
        "run_name",
        "has_best_state",
        "has_history_csv",
        "best_state_path",
    ]
    print(missing[display_cols].to_string(index=False))

# Save the full table
out_path = "model_file_inventory.csv"
df.to_csv(out_path, index=False)
print(f"\nSaved inventory to: {out_path}")

Total expected runs: 136
Complete core runs: 136
Missing/incomplete runs: 0

By category:
                        sum  count
category                          
baseline_full_nodarcy     6      6
official_darcy           40     40
quick_thin_darcy_sweep   90     90

Missing core files:
None. All expected core files were found.

Saved inventory to: model_file_inventory.csv
